# Agent declaration

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from pydantic import BaseModel


tavily_client = TavilyClient()





system_prompt = "You are Chief advisor and your task is find the best dish basing on given order. Information about dish is stored in Recipe that includes" \
"name of the dish, description (what is it? Country of origin?), ingridients list (dict with ingridients as keys and amount of them as float), instruction (explaining how to make a dish)" \
"and serving (amount of servings). To find dish use web search tool."

#Output Structure
class Recipe(BaseModel):
    name:str
    description:str
    ingridients: dict[str, float]
    instruction:str
    serving:int



#Tool kit
@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    
    return tavily_client.search(query)



Chief = create_agent(
    model='gpt-5-nano',
    system_prompt=system_prompt,
    tools=[web_search],
    response_format=Recipe,
    checkpointer=InMemorySaver()
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2354630718.py, line 43)

# Evaluation

In [ ]:
config = {'configurable': {"thread_id":"1"}}

message = 'I have some rice and chciken and I want to prepare some intresting dish from asian cuisine.'
response = Chief.invoke({'messages':[HumanMessage(content=message)]}, config=config)

In [36]:
def print_recipe_dict(recipe_dict: dict):
    # Extract data with defaults to prevent KeyErrors
    name = recipe_dict.get("name", "Unknown Dish").upper()
    description = recipe_dict.get("description", "No description available.")
    ingredients = recipe_dict.get("ingridients", {})
    instructions = recipe_dict.get("instruction", "No instructions provided.")
    servings = recipe_dict.get("serving", 1)

    # Header
    print(f"\n{'*' * 60}")
    print(f"{name.center(60)}")
    print(f"{'*' * 60}")
    
    # Metadata
    print(f"\n[ ORIGIN/TYPE ]  {description}")
    print(f"[ YIELD        ]  {servings} serving(s)")
    
    # Ingredients Table
    print(f"\n{' INGREDIENT ':-<35} | {' AMOUNT ':-<15}")
    for item, amount in ingredients.items():
        # Formatting floats to 2 decimal places for cleaner look
        print(f" {item:<34} | {amount:<14}")
    print(f"{'-' * 52}")

    # Instructions
    print(f"\n[ PREPARATION STEPS ]")
    # Splitting by periods to create a list-style view if it's a long string
    steps = instructions.split('. ')
    for i, step in enumerate(steps, 1):
        if step.strip():
            # Ensure the step ends with a period if it was stripped
            clean_step = step.strip().rstrip('.') + '.'
            print(f"  {i}. {clean_step}")
            
    print(f"\n{'*' * 60}\n")

In [37]:
import json


recipe = response['messages'][-1].content

recipe_dict = json.loads(recipe)
print_recipe_dict(recipe_dict)


************************************************************
               BIBIMBAP WITH CHICKEN BULGOGI                
************************************************************

[ ORIGIN/TYPE ]  A vibrant Korean rice bowl featuring marinated chicken bulgogi, sautéed vegetables, a fried egg, and gochujang sauce. A flavorful, visually striking way to enjoy chicken and rice.
[ YIELD        ]  2 serving(s)

 INGREDIENT ----------------------- |  AMOUNT -------
 cooked_rice_cups                   | 2.0           
 chicken_breast_grams               | 300.0         
 garlic_cloves_minced               | 3.0           
 soy_sauce_tablespoons              | 2.0           
 brown_sugar_teaspoons              | 1.0           
 sesame_oil_teaspoons               | 1.0           
 gochujang_tablespoons              | 2.0           
 sesame_seeds_tablespoons           | 1.0           
 spinach_grams                      | 150.0         
 mushrooms_sliced_grams             | 100.0         
 c

In [38]:
response['messages']

[HumanMessage(content='I have some rice and chciken and I want to prepare some intresting dish from asian cuisine.', additional_kwargs={}, response_metadata={}, id='4baaa751-2cd6-4b07-b23c-2b2b3265b41b'),
 AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 876, 'prompt_tokens': 308, 'total_tokens': 1184, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DfUFYqhzsPrrVzqwNFTHwStwzUSNF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e278e-56bc-7792-a8a0-0601d1777c02-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'Nasi goreng ayam recipe chicken Indonesian fried rice'}, 'id': 'call_3rNItJOqYj